In [1]:
# Script 0: This cell sets up everything needed to connect to Spotify.
# It loads secret keys from the .env file and creates a login URL.
# Open the printed URL, approve access, and copy the code back into your .env file.
import os
from pathlib import Path
import requests
import base64
import pandas as pd
from urllib.parse import urlencode

# Load variables from a local .env file if present
env_path = Path(".env")
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip())

CLIENT_ID = os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")
REDIRECT_URI = "http://127.0.0.1:8888/callback"

if not CLIENT_ID or not CLIENT_SECRET:
    raise ValueError(
        "Missing Spotify credentials. Set SPOTIFY_CLIENT_ID and SPOTIFY_CLIENT_SECRET in your .env file."
    )

params = {
    "client_id": CLIENT_ID,
    "response_type": "code",
    "redirect_uri": REDIRECT_URI,
    "scope": "user-top-read user-read-recently-played user-library-read user-follow-read user-read-private"
}

auth_url = "https://accounts.spotify.com/authorize?" + urlencode(params)
print("Open this URL in your browser:")
print(auth_url)

Open this URL in your browser:
https://accounts.spotify.com/authorize?client_id=0fb96dfd2d4b42c0a595709986258d3b&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8888%2Fcallback&scope=user-top-read+user-read-recently-played+user-library-read+user-follow-read+user-read-private


In [2]:
# Script 1: This cell exchanges the login code for an access token.
# The token is like a temporary key that lets this notebook read your Spotify data.
# If successful, you should see a confirmation message.
auth_code = os.getenv("SPOTIFY_AUTH_CODE", "").strip()
if not auth_code:
    auth_code = input("Paste a fresh Spotify authorization code from the callback URL: ").strip()

if not auth_code:
    raise ValueError("No authorization code provided. Set SPOTIFY_AUTH_CODE in .env or paste one when prompted.")

credentials = base64.b64encode(f"{CLIENT_ID}:{CLIENT_SECRET}".encode()).decode()

token_response = requests.post(
    "https://accounts.spotify.com/api/token",
    headers={
        "Authorization": f"Basic {credentials}",
        "Content-Type": "application/x-www-form-urlencoded"
    },
    data={
        "grant_type": "authorization_code",
        "code": auth_code,
        "redirect_uri": REDIRECT_URI
    }
)

try:
    token_data = token_response.json()
except ValueError:
    raise RuntimeError(
        f"Spotify token endpoint did not return JSON (status {token_response.status_code}). "
        f"Raw response: {token_response.text}"
    )

if "access_token" not in token_data:
    raise RuntimeError(
        "Spotify token request failed. "
        f"Status {token_response.status_code}. Response: {token_data}. "
        "Most common causes: expired/used auth code, redirect URI mismatch, or incorrect client credentials. "
        "Generate a fresh auth code from Script 0, then either update SPOTIFY_AUTH_CODE in .env or paste it when prompted. "
        "Also make sure REDIRECT_URI matches your Spotify app settings exactly."
    )

access_token = token_data["access_token"]
refresh_token = token_data.get("refresh_token")
headers = {"Authorization": f"Bearer {access_token}"}

print("✓ Token obtained successfully")
if not refresh_token:
    print("Note: no refresh token returned. If this was a re-consent flow, Spotify may omit it.")

✓ Token obtained successfully


# My Spotify Wrapped: A PM Analysis
### Using the Spotify Web API to analyze personal listening behavior
**Nico Rodríguez | HCDE 530 | Spring 2026**

In [3]:
# Script 3: This cell checks which Spotify account is currently connected.
# It prints your display name so you can confirm you are logged into the right account.
me = requests.get("https://api.spotify.com/v1/me", headers=headers).json()
print(f"Logged in as: {me['display_name']}")

Logged in as: Nicolás Rodríguez


## Q1: Am I a Loyalist or an Explorer?
*How much overlap exists between my top artists across short, medium, and long term?*

In [4]:
# Script 5: This cell pulls your top 10 artists for three time periods.
# It stores the results and prints each list so we can compare short-term vs long-term taste.
time_ranges = {
    "short_term": "Last 4 Weeks",
    "medium_term": "Last 6 Months",
    "long_term": "All Time"
}

top_artists = {}

for range_key, range_label in time_ranges.items():
    response = requests.get(
        "https://api.spotify.com/v1/me/top/artists",
        headers=headers,
        params={"time_range": range_key, "limit": 10}
    )
    artists = response.json()["items"]
    top_artists[range_key] = [a["name"] for a in artists]
    print(f"\n--- {range_label} ---")
    for i, name in enumerate(top_artists[range_key], 1):
        print(f"  {i}. {name}")


--- Last 4 Weeks ---
  1. Pink Floyd
  2. Yandel
  3. Feid
  4. Radiohead
  5. Omar Courtz
  6. Olivia Dean
  7. Trueno
  8. Harry Styles
  9. Lenny Kravitz
  10. Nsqk

--- Last 6 Months ---
  1. Pink Floyd
  2. Feid
  3. Radiohead
  4. Omar Courtz
  5. Milo j
  6. Mac DeMarco
  7. Olivia Dean
  8. Brent Faiyaz
  9. Yandel
  10. Fontaines D.C.

--- All Time ---
  1. Pink Floyd
  2. Feid
  3. Radiohead
  4. Mac DeMarco
  5. Mac Miller
  6. The Doors
  7. Bad Bunny
  8. Brent Faiyaz
  9. Jeff Buckley
  10. Drake


In [5]:
# Script 6: This cell compares artist overlap across time periods.
# It finds your core artists, new discoveries, and overlap scores.
# Then it labels your listening style (Loyalist, Balanced, or Explorer).
short = set(top_artists["short_term"])
medium = set(top_artists["medium_term"])
long = set(top_artists["long_term"])

short_medium = short & medium
medium_long = medium & long
all_three = short & medium & long
only_short = short - medium - long

print("=== LOYALTY ANALYSIS ===\n")
print(f"Artists in ALL three periods (core taste):")
for a in all_three:
    print(f"  → {a}")

print(f"\nArtists only in last 4 weeks (new discoveries):")
for a in only_short:
    print(f"  → {a}")

print(f"\nOverlap score (short vs medium): {len(short_medium)}/10")
print(f"Overlap score (medium vs long): {len(medium_long)}/10")
print(f"Overlap score (all three): {len(all_three)}/10")

if len(all_three) >= 6:
    profile = "Loyalist — you return to the same artists consistently"
elif len(all_three) >= 3:
    profile = "Balanced — mix of core favorites and new discoveries"
else:
    profile = "Explorer — your taste shifts significantly over time"

print(f"\n🎧 Your listener profile: {profile}")

=== LOYALTY ANALYSIS ===

Artists in ALL three periods (core taste):
  → Pink Floyd
  → Radiohead
  → Feid

Artists only in last 4 weeks (new discoveries):
  → Nsqk
  → Harry Styles
  → Trueno
  → Lenny Kravitz

Overlap score (short vs medium): 6/10
Overlap score (medium vs long): 5/10
Overlap score (all three): 3/10

🎧 Your listener profile: Balanced — mix of core favorites and new discoveries


## Q2: Do I Listen to Classic Catalog or New Music?
*What is the release year distribution of my top tracks across time ranges?*

In [6]:
# Script 8: This cell pulls your top tracks and checks their release years.
# It builds a table of track details and summarizes how old/new your music tends to be.
# This helps answer whether you prefer classic songs or newer releases.
top_tracks = {}

for range_key, range_label in time_ranges.items():
    response = requests.get(
        "https://api.spotify.com/v1/me/top/tracks",
        headers=headers,
        params={"time_range": range_key, "limit": 10}
    )
    tracks = response.json()["items"]
    top_tracks[range_key] = [{
        "name": t["name"],
        "artist": t["artists"][0]["name"],
        "album": t["album"]["name"],
        "release_year": int(t["album"]["release_date"][:4]),
        "duration_sec": round(t["duration_ms"] / 1000),
        "explicit": t["explicit"]
    } for t in tracks]

# Build combined dataframe
rows = []
for range_key, range_label in time_ranges.items():
    for t in top_tracks[range_key]:
        rows.append({**t, "time_range": range_label})

df_tracks = pd.DataFrame(rows)

print("=== RELEASE YEAR DISTRIBUTION ===\n")
for range_label in time_ranges.values():
    subset = df_tracks[df_tracks["time_range"] == range_label]
    avg_year = subset["release_year"].mean()
    oldest = subset["release_year"].min()
    newest = subset["release_year"].max()
    print(f"{range_label}:")
    print(f"  Avg release year: {avg_year:.0f}")
    print(f"  Range: {oldest} → {newest}")
    print()

=== RELEASE YEAR DISTRIBUTION ===

Last 4 Weeks:
  Avg release year: 2020
  Range: 1975 → 2026

Last 6 Months:
  Avg release year: 2019
  Range: 1975 → 2026

All Time:
  Avg release year: 2005
  Range: 1975 → 2025



## Q3: How Do I Actually Consume Music?
*In my recent listening, do I listen through albums or playlists?*

In [ ]:
# Script 11: Export pulled Spotify datasets to one CSV file.
# Run this after Scripts 5, 8, and 10 so all dataframes/dicts exist.
from pathlib import Path

export_frames = []

if "top_artists" in globals():
    artists_rows = []
    for range_key, artists in top_artists.items():
        for rank, artist_name in enumerate(artists, start=1):
            artists_rows.append(
                {
                    "source": "top_artists",
                    "time_range_key": range_key,
                    "rank": rank,
                    "artist": artist_name,
                }
            )
    export_frames.append(pd.DataFrame(artists_rows))

if "df_tracks" in globals():
    tracks_export = df_tracks.copy()
    tracks_export.insert(0, "source", "top_tracks")
    export_frames.append(tracks_export)

if "df_recent" in globals():
    recent_export = df_recent.copy()
    recent_export.insert(0, "source", "recently_played")
    export_frames.append(recent_export)

if not export_frames:
    raise RuntimeError(
        "No pulled data found to export. Run Scripts 5, 8, and 10 first, then run this export cell."
    )

export_df = pd.concat(export_frames, ignore_index=True, sort=False)
output_path = Path("Spotify NR data.csv")
export_df.to_csv(output_path, index=False)

print(f"Saved {len(export_df)} rows to {output_path.resolve()}")

In [7]:
# Script 10: This cell analyzes your 50 most recent listens.
# It checks where your listening came from (album, playlist, etc.) and who you played most.
# This gives a quick view of your real listening behavior lately.
recent_response = requests.get(
    "https://api.spotify.com/v1/me/player/recently-played",
    headers=headers,
    params={"limit": 50}
)

recent_items = recent_response.json()["items"]

rows = []
for item in recent_items:
    track = item["track"]
    context = item.get("context")
    context_type = context["type"] if context else "unknown"
    played_at = item["played_at"]
    
    rows.append({
        "track": track["name"],
        "artist": track["artists"][0]["name"],
        "played_at": played_at,
        "context": context_type,
        "release_year": int(track["album"]["release_date"][:4])
    })

df_recent = pd.DataFrame(rows)
df_recent["played_at"] = pd.to_datetime(df_recent["played_at"])

print("=== LISTENING CONTEXT BREAKDOWN ===\n")
context_counts = df_recent["context"].value_counts()
total = len(df_recent)
for context, count in context_counts.items():
    pct = round(count / total * 100)
    print(f"  {context}: {count} plays ({pct}%)")

print(f"\nMost recent listen: {df_recent.iloc[0]['track']} by {df_recent.iloc[0]['artist']}")
print(f"Oldest in sample: {df_recent.iloc[-1]['track']} by {df_recent.iloc[-1]['artist']}")

print("\n=== TOP ARTISTS IN RECENT PLAYS ===")
print(df_recent["artist"].value_counts().head(5).to_string())

=== LISTENING CONTEXT BREAKDOWN ===

  playlist: 24 plays (48%)
  album: 14 plays (28%)
  unknown: 12 plays (24%)

Most recent listen: The Game of Love by Daft Punk
Oldest in sample: Lose Yourself to Dance (feat. Pharrell Williams) by Daft Punk

=== TOP ARTISTS IN RECENT PLAYS ===
artist
Daft Punk      6
The Strokes    6
Joy Rhodes     2
Feid           2
Central Cee    2
